<a href="https://colab.research.google.com/github/jeffheaton/app_deep_learning/blob/main/t81_558_class_06_1_transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# T81-558: Applications of Deep Neural Networks
**Module 9: Foundations of Generative AI**

* Instructor: [Jeff Heaton](https://sites.washu.edu/jeffheaton/), McKelvey School of Engineering, [Washington University in St. Louis](https://engineering.washu.edu/index.html)
* For more information visit the [class website](https://sites.washu.edu/jeffheaton/t81-558/).

# Module 9 Material

* Part 9.1: Transformer Architecture [[Video]]() [[Notebook]](t81_558_class_09_1_transformers.ipynb)
* **Part 9.2: Pretrained Models and the Hugging Face Ecosystem** [[Video]]() [[Notebook]](t81_558_class_09_2_hugging.ipynb)
* Part 9.3: Embeddings and Vector Representations [[Video]]() [[Notebook]](t81_558_class_09_3_embedding.ipynb)
* Part 9.4: Diffusion Models and Image Generation [[Video]]() [[Notebook]](t81_558_class_09_4_stable_diff.ipynb)
* Part 9.5: Accessing API-Based LLMs [[Video]]() [[Notebook]](t81_558_class_09_5_chat_gpt.ipynb)


# Google CoLab Instructions

The following code ensures that Google CoLab is running the correct version of TensorFlow.
  Running the following code will map your GDrive to ```/content/drive```.

In [1]:
try:
    from google.colab import drive
    COLAB = True
    print("Note: using Google CoLab")
except:
    print("Note: not using Google CoLab")
    COLAB = False

Note: using Google CoLab


# Part 9.2: Pretrained Models and the Hugging Face Ecosystem

In Part 9.1 we looked at the transformer architecture itself: attention, positional encodings, and the encoder/decoder layout that underlies almost every modern large language model. In practice, almost no one trains a transformer of meaningful size from scratch. Pretraining a competitive model costs millions of dollars in GPU time and requires curated datasets measured in trillions of tokens. The dominant workflow in industry and research is instead to **download a pretrained model** that someone else has trained, and then either use it directly or fine-tune it on a smaller, task-specific dataset.

The de facto distribution point for pretrained models is the [Hugging Face Hub](https://huggingface.co). The Hub is essentially "GitHub for models." It hosts hundreds of thousands of model checkpoints, datasets, and demo applications, all accessible through a consistent Python API.

### What you get from the Hugging Face ecosystem

The Hugging Face stack is a collection of open-source Python libraries built around the Hub. The three you will see most often are:

1. **transformers** is the core library. It implements the model architectures (GPT, BERT, Llama, T5, Mistral, and many others) and provides a unified API for loading any compatible checkpoint from the Hub. The `AutoModel` and `AutoTokenizer` classes used below figure out the right architecture from the model's config file so you do not have to write model-specific code.
2. **datasets** provides streaming, memory-mapped access to thousands of NLP, vision, and audio datasets. We will not use it in this notebook but it pairs naturally with `transformers` for fine-tuning workflows.
3. **accelerate** handles device placement, mixed-precision math, and multi-GPU training with very little boilerplate. We install it below because some `transformers` features rely on it.

There are several other libraries in the ecosystem (`peft` for parameter-efficient fine-tuning, `tokenizers` for fast Rust-based tokenization, `diffusers` for image generation models, `evaluate` for metrics) but the three above cover the bulk of what most projects need.

### Causal language models versus other types

The model we load below is a **causal language model**, also called an autoregressive or decoder-only model. Causal models predict the next token given all previous tokens, which is the formulation used by GPT, Llama, Mistral, and similar models. This is the architecture you want for text generation, chatbots, and most "give the model a prompt and let it write" use cases.

This is different from **masked language models** like BERT, which fill in blanked-out tokens and are typically used for classification, embeddings, and sentence-level understanding tasks. It is also different from **encoder-decoder** or **sequence-to-sequence** models like T5 and BART, which read an input sequence and produce an output sequence and are common in translation and summarization.

The class `AutoModelForCausalLM` tells the `transformers` library to load the checkpoint configured for next-token prediction. There are parallel `AutoModelFor...` classes for the other model types.

### About SmolLM2-135M

For this demonstration we use [SmolLM2-135M-Instruct](https://huggingface.co/HuggingFaceTB/SmolLM2-135M-Instruct), a small instruction-tuned model from the Hugging Face team. At roughly 135 million parameters it is tiny by modern standards (production chatbots are typically 7 billion to several hundred billion parameters), but small models are useful for teaching because they:

* Load quickly, even on a free Colab tier or a CPU.
* Demonstrate the full generation pipeline without requiring a powerful GPU.
* Make the limitations of small models visible, which helps build intuition about why bigger models behave differently.

The `-Instruct` suffix indicates this checkpoint has been fine-tuned to follow instructions in a chat-style format. The base SmolLM2-135M model also exists on the Hub and would simply continue text rather than respond to prompts.


### Loading a Pretrained Model

The cell below performs the standard "load a Hugging Face model" sequence. It is worth walking through line by line because almost every script that uses the `transformers` library begins with some variant of this code.

**Installing the libraries.** The `!pip` line installs the latest versions of `transformers` and `accelerate`. The leading `!` runs the command in the notebook's shell. The `-q` flag suppresses pip's verbose output, and `-U` upgrades the packages if older versions are already installed. In a fresh Colab environment this typically completes in 10 to 20 seconds.

**Selecting the device.** PyTorch needs to know whether to place tensors on the GPU or the CPU. The line `torch.device("cuda" if torch.cuda.is_available() else "cpu")` picks GPU when one is present and falls back to CPU otherwise. Generation on a 135M-parameter model is feasible on CPU but noticeably slower, so a GPU runtime in Colab is preferred.

**Identifying the model.** The `model_name` variable is the path to the checkpoint on the Hugging Face Hub, in the format `{organization}/{model}`. The first time you load a model, `transformers` downloads the weights and config files and caches them locally (typically under `~/.cache/huggingface`). Subsequent runs reuse the cached copy.

**Loading the tokenizer.** `AutoTokenizer.from_pretrained(model_name)` downloads the tokenizer associated with this specific checkpoint. The tokenizer is responsible for converting text into the integer token IDs the model actually consumes, and for converting generated IDs back into text. It is essential that the tokenizer match the model: a model trained with one vocabulary will produce nonsense if you feed it tokens from a different vocabulary.

**Loading the model with the right precision.** `AutoModelForCausalLM.from_pretrained(...)` downloads the weights and instantiates the model. The `torch_dtype` argument controls the numeric precision of the weights. On a GPU we use `float16` (half precision), which cuts memory use roughly in half and is well supported by modern NVIDIA hardware. On CPU we fall back to `float32` because most CPUs do not have efficient half-precision math. For larger models you will also see `bfloat16` and 8-bit or 4-bit quantization, which trade further accuracy for memory savings.

**Moving to the device and switching to eval mode.** `.to(device)` transfers the model parameters to the chosen device. `model.eval()` disables training-only behaviors like dropout. Forgetting `eval()` is a common bug that makes outputs nondeterministic in subtle ways even when sampling is turned off.


In [2]:
!pip -q install -U transformers accelerate

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model_name = "HuggingFaceTB/SmolLM2-135M-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
).to(device)

model.eval()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 98.1 MB/s eta 0:00:00
Using device: cuda


config.json:   0%|          | 0.00/861 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/269M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(49152, 576, padding_idx=2)
    (layers): ModuleList(
      (0-29): 30 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=576, out_features=576, bias=False)
          (k_proj): Linear(in_features=576, out_features=192, bias=False)
          (v_proj): Linear(in_features=576, out_features=192, bias=False)
          (o_proj): Linear(in_features=576, out_features=576, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=576, out_features=1536, bias=False)
          (up_proj): Linear(in_features=576, out_features=1536, bias=False)
          (down_proj): Linear(in_features=1536, out_features=576, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((576,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((576,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((576,), eps=1e-05)
    (r

### Generating Text from a Prompt

With the model loaded, we can now run inference. The cell below sends a prompt through the model and prints the response. Each step in this cell is doing real work, so it is worth understanding what happens.

**Building the prompt with a chat template.** Modern instruction-tuned models do not just take raw text. They expect prompts wrapped in a specific structure that distinguishes user turns from assistant turns and from system instructions. The exact format varies by model: Llama uses one set of special tokens, Mistral uses another, ChatML uses a third, and so on. Rather than hard-coding any of these, `tokenizer.apply_chat_template` reads the format from the tokenizer's metadata and assembles the right string for whichever model you loaded. Passing `add_generation_prompt=True` appends the special tokens that signal "now it is the assistant's turn to write," which is what tells the model to begin its response.

**Tokenizing.** `tokenizer(input_text, return_tensors="pt")` converts the formatted string into a PyTorch tensor of integer token IDs. The `.to(device)` call moves that tensor to the GPU (or CPU) so it lives on the same device as the model.

**Running generation.** `model.generate(...)` is the workhorse. Several arguments deserve attention:

* `max_new_tokens=120` caps how long the response can be. This counts only the generated tokens, not the prompt.
* `do_sample=True` turns on stochastic sampling. With this off, the model would do greedy decoding (always pick the single highest-probability token), which produces repetitive, deterministic output.
* `temperature=0.7` scales the logits before sampling. Lower values (closer to 0) make the model more confident and more deterministic. Higher values (above 1.0) make it more random and creative. 0.7 is a common middle-ground default.
* `top_p=0.9` enables nucleus sampling: at each step the model only samples from the smallest set of tokens whose cumulative probability exceeds 0.9. This prunes very low-probability tokens that would otherwise occasionally produce strange output.
* `pad_token_id=tokenizer.eos_token_id` is a small detail that suppresses a warning. SmolLM2's tokenizer does not define a separate pad token, so we tell `generate` to use the end-of-sequence token for padding.

The `with torch.no_grad():` block disables gradient tracking. Gradients are only needed during training; turning them off during inference saves memory and runs faster.

**Decoding the response.** `model.generate` returns a tensor that contains the full sequence: prompt tokens followed by generated tokens. The slice `output_ids[0][inputs["input_ids"].shape[-1]:]` selects only the newly generated tokens by skipping past the length of the input. Then `tokenizer.decode(..., skip_special_tokens=True)` converts those token IDs back into a readable string and strips the chat-template special tokens so you get clean text.

**A note on output quality.** Because we are using a 135M-parameter model, do not be surprised if the response is brief, occasionally awkward, or sometimes wrong on facts. This is expected at this scale. Swapping `model_name` for a larger checkpoint (for instance `HuggingFaceTB/SmolLM2-1.7B-Instruct` or a 7B-parameter model) and rerunning will produce noticeably better answers, at the cost of more memory and slower generation.


In [3]:
# Cell #3
prompt = "Explain neural networks in one short paragraph."

messages = [
    {"role": "user", "content": prompt}
]

input_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(input_text, return_tensors="pt").to(device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=120,
        temperature=0.7,
        do_sample=True,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )

response = tokenizer.decode(
    output_ids[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True,
)

print(response)

Neural networks are mathematical models inspired by the structure and function of the human brain. They consist of interconnected nodes or "neurons" that process and transmit information, much like the billions of neurons in the human brain. These nodes are connected to each other using a network of "synapses," which allow the network to learn and adapt. The basic idea behind neural networks is to create a set of interconnected nodes, each with a specific function, that can be combined and generalized to make predictions or classify data.


### Summary and Things to Try

The pattern in this notebook is the foundation for nearly all generative AI work outside of API-only services like the OpenAI API or Anthropic's Claude API (which we cover in Part 9.5). The full sequence is:

1. Pick a model from the Hugging Face Hub.
2. Load the matching tokenizer and model with `AutoTokenizer` and `AutoModelForCausalLM`.
3. Format the prompt with `apply_chat_template` for instruction-tuned models.
4. Call `model.generate` with sampling parameters tuned for your use case.
5. Slice off the prompt tokens and decode the rest.

A few experiments that will sharpen your intuition:

* **Change the prompt** to something open-ended ("Write a short story about a dragon learning Python") and see how the same model behaves with creative versus factual requests.
* **Adjust `temperature`** between 0.1 and 1.5 and observe how the output character changes. Very low values produce flat, predictable text. Very high values produce incoherent text.
* **Set `do_sample=False`** to switch to greedy decoding. Notice that generation becomes deterministic: rerunning the cell gives the same output every time.
* **Swap the model**. Try `HuggingFaceTB/SmolLM2-360M-Instruct` or `HuggingFaceTB/SmolLM2-1.7B-Instruct` and compare quality. The code does not change, only the `model_name` string.
* **Add a system message** by prepending `{"role": "system", "content": "You are a helpful tutor who explains things to a beginner."}` to the `messages` list, and watch how the response style shifts.

In Part 9.3 we move from generating text to extracting **embeddings** from these models: dense vector representations that capture semantic meaning and that power search, retrieval-augmented generation (RAG), and clustering applications.
